In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv("spam.csv")

In [23]:
df.head()

,Message,output
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


In [3]:
le = LabelEncoder()

In [4]:
df["output"] = le.fit_transform(df.Category)

In [6]:
df.head()

,Category,Message,output
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [5]:
df.drop(["Category"], inplace=True, axis=1)
df.head()

,Message,output
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


In [6]:
X = df[['Message']]  # Features: message text
y = df['output']     # Labels: spam (1) or ham (0)

In [8]:
# Debug: Check data structure
print("X shape:", X.shape)
print("y shape:", y.shape)
print("y value counts:")
print(y.value_counts())
print("\nFirst y values:", y.head(10).values)

X shape: (5572, 1)
y shape: (5572,)
y value counts:
output
0    4825
1     747
Name: count, dtype: int64

First y values: [0 0 1 0 0 1 0 0 1 1]


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33)

In [10]:
v = CountVectorizer()
X_train.head()

,Message
2210,"Hmm well, night night"
1578,Remember all those whom i hurt during days of ...
1485,"Sorry, I'll call later"
1725,There bold 2 &lt;#&gt; . Is that yours
2446,The guy (kadeem) hasn't been selling since the...


In [11]:
type(X_train)

pandas.DataFrame

In [12]:
X_train_count = v.fit_transform(X_train.Message)
X_train_count.toarray()[:2]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(2, 7140))

In [9]:
v = CountVectorizer()

X_train_count = v.fit_transform(X_train.Message)
bernouli = BernoulliNB()
bernouli.fit(X_train_count, y_train)

X_test_count = v.transform(X_test.Message)
bernouli.score(X_test_count, y_test)

0.9722675367047309

In [30]:
# Debug BernoulliNB
print("X_train_count shape:", X_train_count.shape)
print("y_train shape:", y_train.shape)
print("y_train value counts:\n", y_train.value_counts())
print("\nX_test_count shape:", X_test_count.shape)  
print("y_test unique values:", y_test.unique())

X_train_count shape: (3733, 7061)
y_train shape: (3733,)
y_train value counts:
 Message
Sorry, I'll call later                                                                                                                                 16
I cant pick the phone right now. Pls send a message                                                                                                     7
Ok...                                                                                                                                                   7
Please call our customer service representative on FREEPHONE 0808 145 4742 between 9am-11pm as you have WON a guaranteed £1000 cash or £5000 prize!     4
Ok                                                                                                                                                      4
                                                                                                                                              

In [10]:
v = CountVectorizer()

X_train_count = v.fit_transform(X_train.Message)
multin = MultinomialNB()
multin.fit(X_train_count, y_train)

X_test_count = v.transform(X_test.Message)
multin.score(X_test_count, y_test)

0.9842305600870038

In [32]:
# Check what MultinomialNB is predicting
print("Sample predictions from MultinomialNB:")
sample_predictions = multin.predict(X_test_count[:10])
print("Predictions:", sample_predictions)
print("Actual:", y_test.values[:10])
print("\nClass distribution in predictions:")
from collections import Counter
pred_counts = Counter(sample_predictions)
print(dict(pred_counts))

Sample predictions from MultinomialNB:
Predictions: ["Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"
 "The last thing i ever wanted to do was hurt you. And i didn't think it would have. You'd laugh, be embarassed, delete the tag and keep going. But as far as i knew, it wasn't even up. The fact that you even felt like i would do it to hurt you shows you really don't know me at all. It was messy wednesday, but it wasn't bad. The problem i have with it is you HAVE the time to clean it, but you choose not to. You skype, you take pictures, you sleep, you want to go out. I don't mind a few things here and there, but when you don't make the bed, when you throw laundry on top of it, when i can't have a friend in the house because i'm embarassed that there's underwear and bras strewn on the bed, pillows on the floor, that's something else. You used to be good about at least making the b

## Pipeline

In [11]:
clf = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('nb', BernoulliNB())
])
clf.fit(X_train.Message, y_train)
clf.score(X_test.Message, y_test)

0.9722675367047309